In [1]:
# Importing Modules
import torch
import pandas as pd
import numpy as np
import itertools
from src.dataloader import loadData
from src.model import SimpleGCN, SimpleGAT, SimpleGIN, SimpleGraphSAGE
from src.train_test import TrainGNN, TestGNN
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

In [2]:
######################## DATA-1 ##################################
# Loading ESOL data
esol_train_data = pd.read_csv("data/train/ESOL.csv")
esol_test_data = pd.read_csv("data/test/ESOL.csv")

######################## DATA-2 ##################################
# Loading FreeSolv data
freeSolv_train_data = pd.read_csv("data/train/FreeSolv.csv")
freeSolv_test_data = pd.read_csv("data/test/FreeSolv.csv")

######################## DATA-3 ##################################
# Loading Lipophilicity data
lipophilicity_train_data = pd.read_csv("data/train/Lipophilicity.csv")
lipophilicity_test_data = pd.read_csv("data/test/Lipophilicity.csv")

In [3]:
# Models
model_dict = {
		"SimpleGCN":SimpleGCN, 
		"SimpleGAT":SimpleGAT, 
		"SimpleGIN":SimpleGIN,
		"SimpleGraphSAGE":SimpleGraphSAGE
}

# Models Hyperparameters
model_hyperparams_dict = {
	"SimpleGCN":[0.01, 64, 8],
	"SimpleGAT":[0.005, 128, 32],
	"SimpleGIN":[0.001, 64, 4],
	"SimpleGraphSAGE":[0.005, 128, 4]
}

# Function to run model training
def model_training(model, X_train, y_train, b_size, lr):
	loader = loadData(X_train, y_train, batch_size=b_size)
	model = TrainGNN(model, loader, epochs=100, learning_rate=lr)
	return model

# Function to run model testing
def model_testing(model, X_test, y_test, b_size):
	loader = loadData(X_test, y_test, batch_size=b_size, shuf=False)
	y_test, y_pred = TestGNN(model, loader)
	y_test = np.asarray(y_test, dtype=float)
	y_pred = np.asarray(y_pred, dtype=float)
	return y_test, y_pred

# Function for weights optimization
def weights_optim(y_pred1, y_pred2, y_true):
	weights = np.linspace(0, 1, 101)
	best_w, best_mae = None, float("inf")
	for w in weights:
		y_ens = w * y_pred1 + (1 - w) * y_pred2
		mae = mean_absolute_error(y_true, y_ens)
		if mae < best_mae:
			best_mae = mae
			best_w = w
	return best_w


# Function to run analysis
def RunGNN(train_data, test_data, dataName, modelName1, modelName2):

	# Train val split
	X_train, X_val, y_train, y_val = train_test_split(train_data["smiles"], train_data["target"], test_size=0.2, random_state=42)

	# Training data and target labels
	X_train = X_train.to_numpy()
	y_train = y_train.to_numpy()
    
	# Validation data and target labels
	X_val = X_val.to_numpy()
	y_val = y_val.to_numpy()

	# Testing data and target labels
	smiles_X_test = test_data["smiles"]
	X_test = smiles_X_test.to_numpy()
	y_test = test_data["target"]
	y_test = y_test.to_numpy()

	# Params
	lr1, h_dim1, b_size1 = model_hyperparams_dict[modelName1]
	lr2, h_dim2, b_size2 = model_hyperparams_dict[modelName2]

	# Model
	model1 = model_dict[modelName1](num_features=8, hidden_dim=h_dim1, num_classes=1)
	model2 = model_dict[modelName2](num_features=8, hidden_dim=h_dim2, num_classes=1)

	# Model training
	model1 = model_training(model1, X_train, y_train, b_size1, lr1)
	model2 = model_training(model2, X_train, y_train, b_size2, lr2)

	# Model Validation | Weights Optimization
	y_val1, y_pred1 = model_testing(model1, X_val, y_val, b_size1)
	y_val2, y_pred2 = model_testing(model2, X_val, y_val, b_size2)
	w = weights_optim(y_pred1, y_pred2, y_val1)

	# Model Testing
	y_test1, y_pred1 = model_testing(model1, X_test, y_test, b_size1)
	y_test2, y_pred2 = model_testing(model2, X_test, y_test, b_size2)
	y_ensemble = (w*y_pred1)+((1-w)*y_pred2)

	# Calculate MAE
	mae = mean_absolute_error(y_test1, y_ensemble)

	# Calculate RMSE
	rmse = root_mean_squared_error(y_test1, y_ensemble)
    
	# Calculate pearson r and p_val
	r, p_val = pearsonr(y_test1, y_ensemble)
    
	# Return results
	return pd.DataFrame({"Data":[dataName], "Model1":[modelName1],"Model2":[modelName2],
			"MAE":round(mae, 2), "RMSE":[round(rmse, 2)], 
			"r":[round(r, 2)], "p-val":[round(p_val, 3)]})

In [4]:
# Data dict
datasets = {
    "ESOL":{"train":esol_train_data, "test":esol_test_data},
    "FreeSolv":{"train":freeSolv_train_data, "test":freeSolv_test_data},
    "Lipophilicity":{"train":lipophilicity_train_data, "test":lipophilicity_test_data}
}

In [5]:
# List to store results
temp_out = []

model_pairs = (list(combinations(list(model_dict.keys()), 2))+[(m, m) for m in list(model_dict.keys())])

# Model loop
for models in model_pairs:
	modelName1, modelName2 = models
	# Data loop
	for dataName, data in datasets.items():
		# Run Analysis for model and dataset
		temp_out.append(RunGNN(data["train"], data["test"], dataName, modelName1, modelName2))

# Final output
GNN_out = pd.concat(temp_out, ignore_index=True)
GNN_out.to_csv("results/Output_Ensemble_GNN_Analysis.csv")
GNN_out

,Data,Model1,Model2,MAE,RMSE,r,p-val
0,ESOL,SimpleGCN,SimpleGAT,0.99,1.29,0.82,0.0
1,FreeSolv,SimpleGCN,SimpleGAT,1.99,2.90,0.75,0.0
2,Lipophilicity,SimpleGCN,SimpleGAT,0.84,1.09,0.56,0.0
3,ESOL,SimpleGCN,SimpleGIN,1.17,1.45,0.79,0.0
4,FreeSolv,SimpleGCN,SimpleGIN,2.26,2.95,0.69,0.0
5,Lipophilicity,SimpleGCN,SimpleGIN,0.89,1.10,0.43,0.0
6,ESOL,SimpleGCN,SimpleGraphSAGE,0.97,1.27,0.81,0.0
7,FreeSolv,SimpleGCN,SimpleGraphSAGE,1.55,2.09,0.87,0.0
8,Lipophilicity,SimpleGCN,SimpleGraphSAGE,0.84,1.04,0.53,0.0
9,ESOL,SimpleGAT,SimpleGIN,1.07,1.35,0.80,0.0


In [6]:
# Function to plot ensemble analysis results
def plot_ensemble_hmap(df, dataset_name):
	# Subsetting data
	data = df[df["Data"] == dataset_name].reset_index(drop=True)
	models = sorted(set(data["Model1"]).union(data["Model2"]))
	mat = pd.DataFrame(np.nan, index=models, columns=models)
	for _, row in data.iterrows():
		m1, m2, v = row["Model1"], row["Model2"], row["RMSE"]
		mat.loc[m1, m2] = v
		mat.loc[m2, m1] = v
	# Masking
	mat_plot = mat.loc[models[::-1], models]
	mask = np.fliplr(np.tril(np.ones_like(mat, dtype=bool), k=-1))
	# Plotting Masked Heatmap
	sns.set_theme(style="white", context="paper")
	ax = sns.heatmap(mat_plot, mask=mask, annot=True, fmt=".3f",
		cmap="viridis_r", square=True, linewidths=0.5, cbar_kws={"label": "RMSE"})
	ax.xaxis.tick_top()
	ax.xaxis.set_label_position("top")
	plt.xticks(rotation=90, ha="left")
	plt.yticks(rotation=0)
	plt.tight_layout()
	plt.savefig(f"results/GNN_Ensemble_Analysis_{dataset_name}_Dataset_Plot.png", dpi=300)
	plt.close()

In [7]:
# Plot for ESOL
plot_ensemble_hmap(GNN_out, "ESOL")

In [8]:
# Plot for FreeSolv
plot_ensemble_hmap(GNN_out, "FreeSolv")

In [9]:
# Plot for Lipophilicity
plot_ensemble_hmap(GNN_out, "Lipophilicity")